In [6]:
# combine 39 batches of data in /opt/dlami/nvme/TIR/simpletir_maxres8000_maxpro16000_maxturn5_simplelr_math_35_train_deepscaler_train_Qwen2.5-7B_mis_cliph2.0_step400/step_records/
from datasets import load_dataset
from datasets import concatenate_datasets

ds_all = []
for i in range(1,40):
    ds = load_dataset("json", data_files=f"/opt/dlami/nvme/TIR/simpletir_maxres8000_maxpro16000_maxturn5_simplelr_math_35_train_deepscaler_train_Qwen2.5-7B_mis_cliph2.0_step400/step_records/step-val-{i}.json", split="train")
    ds_all.append(ds)

ds_all = concatenate_datasets(ds_all)

In [2]:
len(ds_all)

312544

In [3]:
ds_all[0]

{'data_source': 'deepscaler/sft.parquet',
 'prompt': 'system\nYou are a helpful assistant.\nuser\nSolve the following problem step by step. You now have the ability to selectively write executable Python code to enhance your reasoning process. The Python code will be executed by an external sandbox, and the output (after "Code execution result: ") is returned to aid your reasoning and help you arrive at the final answer. The Python code should be complete scripts, including necessary imports.\n\nCode Format:\nEach code snippet is wrapped between ```. You need to use `print()` to output intermediate results.\n\nAnswer Format:\nYou must use \\boxed to return your answer. The last part of your response should be:\n\\boxed{\'The final answer goes here.\'}\n\nUser Question:\nAutomobile license plates for a state consist of four letters followed by a dash and two single digits. How many different license plate combinations are possible if exactly one letter is repeated exactly once, but digi

In [7]:
# filter out score == 0
ds_all = ds_all.filter(lambda x: x["score"] != 0 and x["extra_info"]["valid_code"] != 0)
len(ds_all)

190550

# NEW SECTION: 单轮正确样本筛选与去重


In [9]:
# Step 1: 筛选只有一轮且tool call正确的样本
import re

def extract_code_blocks(response):
    """提取所有代码块（在 ```python 或 ``` 和 ``` 之间）"""
    # 匹配 ```python...``` 或 ```...```
    pattern = r'```(?:python|py)?\n(.*?)\n```'
    code_blocks = re.findall(pattern, response, re.DOTALL)
    return code_blocks

def extract_observations(response):
    """提取所有observations（在 'Code execution result:' 和换行之间）"""
    # 匹配 "Code execution result:" 后面的内容直到遇到两个连续换行或字符串结束
    pattern = r'Code execution result:(.*?)(?=\n\n|\n[^\n]|$)'
    observations = re.findall(pattern, response, re.DOTALL)
    # 去掉每个observation前后的空白
    observations = [obs.strip() for obs in observations]
    return observations

def is_single_turn_correct(example):
    """检查是否只有一轮且tool call正确"""
    observations = extract_observations(example["response"])
    code_blocks = extract_code_blocks(example["response"])
    if len(observations) != 1 or len(code_blocks) != 1:
        return False
    if "err" in observations[0].lower() or "timeout" in observations[0].lower() or observations[0].strip() == "[]" or observations[0].strip().lower() == "none":
        return False
    
    return True

# 应用筛选
single_turn_correct = ds_all.filter(is_single_turn_correct)
print(f"原始样本数: {len(ds_all)}")
print(f"单轮正确样本数: {len(single_turn_correct)}")


原始样本数: 190550
单轮正确样本数: 149977


In [10]:
# Step 2: 截断第一个代码块之后的内容

def truncate_after_first_code_block(example):
    """保留第一个代码块及其之前的内容，截断代码块之后的所有内容"""
    response = example["response"]
    
    # 匹配代码块 ```python ... ``` 或 ```py ... ``` 或 ``` ... ```
    # 使用非贪婪匹配 .*? 配合 re.DOTALL
    code_pattern = r'```(?:python|py)?\n.*?\n```'
    match = re.search(code_pattern, response, re.DOTALL)
    
    if match:
        # 截取到第一个代码块结束的位置
        end_pos = match.end()
        example["response"] = response[:end_pos]
    
    return example

# 应用截断
truncated_data = single_turn_correct.map(truncate_after_first_code_block)
print(f"截断后样本数: {len(truncated_data)}")


Map: 100%|██████████| 149977/149977 [00:18<00:00, 8193.98 examples/s]

截断后样本数: 149977


In [11]:
# Step 3: 提取code blocks并基于相似度去重

def extract_code_blocks(response):
    """提取所有代码块（在 ```python 或 ``` 和 ``` 之间）"""
    # 匹配 ```python...``` 或 ```...```
    pattern = r'```(?:python)?\n(.*?)\n```'
    code_blocks = re.findall(pattern, response, re.DOTALL)
    return code_blocks

def compute_code_similarity(code1, code2):
    """计算两段代码的相似度（基于字符）"""
    from difflib import SequenceMatcher
    return SequenceMatcher(None, code1, code2).ratio()

# 提取所有样本的code blocks
samples_with_code = []
for i in range(len(truncated_data)):
    example = truncated_data[i]
    code_blocks = extract_code_blocks(example['response'])
    if code_blocks:
        # 合并所有code blocks
        combined_code = '\n\n'.join(code_blocks)
        samples_with_code.append({
            'index': i,
            'example': example,
            'code': combined_code
        })

print(f"包含代码的样本数: {len(samples_with_code)}")


包含代码的样本数: 149977


In [12]:
# Step 4: 去重 - 每个prompt保留最多3个多样性最高的样本

from tqdm import tqdm
from collections import defaultdict

# 按prompt分组
prompt_to_samples = defaultdict(list)
for idx, sample_info in enumerate(samples_with_code):
    prompt = sample_info['example']['prompt']
    prompt_to_samples[prompt].append((idx, sample_info))

# 对每个prompt组内进行去重
keep_indices = set()
filtered_count = 0
max_samples_per_prompt = 3
similarity_threshold = 0.90

for prompt, samples_list in tqdm(prompt_to_samples.items(), desc="Processing prompts"):
    if len(samples_list) <= max_samples_per_prompt:
        # 样本数不超过3个，全部保留
        for idx, _ in samples_list:
            keep_indices.add(idx)
        continue
    
    # 贪心选择：每次选择与已选样本相似度最小的样本
    local_keep = []
    
    # 随机选择第一个样本作为起点
    first_idx, first_sample = samples_list[0]
    local_keep.append((first_idx, first_sample))
    keep_indices.add(first_idx)
    
    # 继续选择剩余样本
    remaining = samples_list[1:]
    
    while len(local_keep) < max_samples_per_prompt and remaining:
        # 计算每个候选样本与已选样本的最小相似度
        best_candidate = None
        max_min_similarity = -1  # 我们想要最小化最大相似度
        
        for candidate_idx, candidate_sample in remaining:
            # 计算该候选与所有已选样本的相似度，取最大值
            max_similarity = 0
            for kept_idx, kept_sample in local_keep:
                similarity = compute_code_similarity(
                    candidate_sample['code'],
                    kept_sample['code']
                )
                max_similarity = max(max_similarity, similarity)
            
            # 如果这个候选的"最大相似度"比当前最佳还小，更新最佳候选
            if best_candidate is None or max_similarity < max_min_similarity:
                best_candidate = (candidate_idx, candidate_sample)
                max_min_similarity = max_similarity
        
        # 检查最佳候选是否满足阈值
        if max_min_similarity < similarity_threshold:
            local_keep.append(best_candidate)
            keep_indices.add(best_candidate[0])
            remaining.remove(best_candidate)
        else:
            # 所有剩余样本都与已选样本太相似，停止
            break
    
    filtered_count += len(samples_list) - len(local_keep)

# 收集最终保留的样本
final_samples = [samples_with_code[i]['example'] for i in sorted(keep_indices)]

print(f"\n去重统计:")
print(f"  原始样本数: {len(samples_with_code)}")
print(f"  保留样本数: {len(final_samples)}")
print(f"  过滤样本数: {len(samples_with_code) - len(final_samples)}")
print(f"  不同prompt数: {len(prompt_to_samples)}")
print(f"  平均每个prompt保留: {len(final_samples) / len(prompt_to_samples):.2f} 个样本")


Processing prompts: 100%|██████████| 7503/7503 [04:04<00:00, 30.63it/s]


去重统计:
  原始样本数: 149977
  保留样本数: 21196
  过滤样本数: 128781
  不同prompt数: 7503
  平均每个prompt保留: 2.83 个样本


In [13]:
# 统计每个prompt保留的样本数量分布
from collections import Counter

prompt_sample_counts = defaultdict(int)
prompt_to_kept_samples = defaultdict(list)

for idx in keep_indices:
    sample = samples_with_code[idx]['example']
    prompt = sample['prompt']
    prompt_sample_counts[prompt] += 1
    prompt_to_kept_samples[prompt].append(sample)

# 统计分布
count_distribution = Counter(prompt_sample_counts.values())
print("每个prompt保留的样本数量分布:")
for count, num_prompts in sorted(count_distribution.items()):
    print(f"  保留 {count} 个样本: {num_prompts} 个prompts")

# 找到保留样本最多的prompt
max_samples_prompt = max(prompt_sample_counts.items(), key=lambda x: x[1])
max_prompt = max_samples_prompt[0]
max_count = max_samples_prompt[1]

print(f"\n保留样本最多的prompt (保留了 {max_count} 个样本):")
print("=" * 100)
print(f"\nPrompt内容:\n{max_prompt[:500]}...")
print("\n" + "=" * 100)

print(f"\n该prompt下保留的样本:")
for i, sample in enumerate(prompt_to_kept_samples[max_prompt][:3]):  # 只显示前3个
    print(f"\n--- 样本 {i+1} ---")
    print(f"Response (前300字符):\n{sample['response']}...")
    print("-" * 80)


每个prompt保留的样本数量分布:
  保留 1 个样本: 481 个prompts
  保留 2 个样本: 351 个prompts
  保留 3 个样本: 6671 个prompts

保留样本最多的prompt (保留了 3 个样本):

Prompt内容:
system
You are a helpful assistant.
user
Solve the following problem step by step. You now have the ability to selectively write executable Python code to enhance your reasoning process. The Python code will be executed by an external sandbox, and the output (after "Code execution result: ") is returned to aid your reasoning and help you arrive at the final answer. The Python code should be complete scripts, including necessary imports.

Code Format:
Each code snippet is wrapped between ```. You...


该prompt下保留的样本:

--- 样本 1 ---
Response (前300字符):
To solve this problem, let's break it down step-by-step:

1. **Generate the combinations for the letters:**
    - The license plate format requires four letters, with exactly one of those letters being repeated exactly once.
    - The number of ways to choose which letter will be the repeated letter is 26 (sinc

In [14]:
# Step 5: 转换为SFT格式

new_sft_data = []

for sample in final_samples:
    # 提取system prompt和user prompt
    system_match = re.search(r'^(.*?)\nuser\n', sample['prompt'], re.DOTALL)
    user_match = re.search(r'user\n(.*?)(?:\nassistant)?$', sample['prompt'], re.DOTALL)
    
    if user_match:
        system_prompt = system_match.group(1).strip() if system_match else ""
        user_prompt = user_match.group(1).strip()
        
        messages = []
        if system_prompt:
            messages.append({"role": "system", "content": system_prompt})
        messages.append({"role": "user", "content": user_prompt})
        messages.append({"role": "assistant", "content": sample['response']})
        
        new_sft_data.append({"messages": messages})

print(f"最终SFT数据样本数: {len(new_sft_data)}")


最终SFT数据样本数: 21196


In [15]:
# 查看一个样本示例
new_sft_data[0]


{'messages': [{'role': 'system',
   'content': 'system\nYou are a helpful assistant.'},
  {'role': 'user',
   'content': 'Solve the following problem step by step. You now have the ability to selectively write executable Python code to enhance your reasoning process. The Python code will be executed by an external sandbox, and the output (after "Code execution result: ") is returned to aid your reasoning and help you arrive at the final answer. The Python code should be complete scripts, including necessary imports.\n\nCode Format:\nEach code snippet is wrapped between ```. You need to use `print()` to output intermediate results.\n\nAnswer Format:\nYou must use \\boxed to return your answer. The last part of your response should be:\n\\boxed{\'The final answer goes here.\'}\n\nUser Question:\nAutomobile license plates for a state consist of four letters followed by a dash and two single digits. How many different license plate combinations are possible if exactly one letter is repeate

In [17]:
# save to jsonl
import json
with open("filtered_sft_data.jsonl", "w") as f:
    for sample in new_sft_data:
        f.write(json.dumps(sample) + "\n")

# OLD SECTION: 原始的多轮数据处理流程


In [5]:
# 1. keep response with no error and no duplicate code/observations
import re

def extract_code_blocks(response):
    """提取所有代码块（在 ```python 或 ``` 和 ``` 之间）"""
    # 匹配 ```python...``` 或 ```...```
    pattern = r'```(?:python)?\n(.*?)\n```'
    code_blocks = re.findall(pattern, response, re.DOTALL)
    return code_blocks

def extract_observations(response):
    """提取所有observations（在 'Code execution result:' 和换行之间）"""
    # 匹配 "Code execution result:" 后面的内容直到遇到两个连续换行或字符串结束
    pattern = r'Code execution result:(.*?)(?=\n\n|\n[^\n]|$)'
    observations = re.findall(pattern, response, re.DOTALL)
    # 去掉每个observation前后的空白
    observations = [obs.strip() for obs in observations]
    return observations

def is_valid_response(response):
    """
    检查response是否有效：
    1. 内部不能有重复的代码块
    2. 内部不能有重复的observations
    3. 最后一个observation不能包含error, timeout, 空或None
    """
    # 提取代码块和observations
    code_blocks = extract_code_blocks(response)
    observations = extract_observations(response)
    
    # 如果没有observations，认为是无效的
    if len(observations) == 0:
        return False
    
    # 1. 检查代码块是否有重复
    if len(code_blocks) != len(set(code_blocks)):
        return False
    
    # 2. 检查observations是否有重复
    if len(observations) != len(set(observations)):
        return False
    
    # 3. 检查最后一个observation
    last_obs = observations[-1].lower()
    if any(keyword in last_obs for keyword in ['err', 'timeout']):
        return False
    
    # 检查是否为空或None
    if last_obs.strip() == '' or last_obs.strip().lower() == 'none' or 'error' in last_obs.strip().lower() or 'timeout' in last_obs.strip().lower():
        return False
    
    return True

ds_no_error = ds_all.filter(lambda x: is_valid_response(x["response"]))
len(ds_no_error)



179574

In [6]:
from collections import defaultdict

prompt2response_no_error = defaultdict(list)
for item in ds_no_error:
    prompt2response_no_error[item["extra_info"]['index']].append({'response':item["response"], 'score':item["score"], 'prompt':item["prompt"]})

len(prompt2response_no_error)

7904

In [8]:
# 处理有错误observation的response，重组prompt和response
import re

def has_error_in_observation(obs):
    """检查observation是否包含错误或失败"""
    obs_lower = obs.lower()
    return any(keyword in obs_lower for keyword in ['err', 'error', 'timeout', 'exception', 'traceback']) or obs.strip() == '' or obs.strip().lower() == 'none'

def reorganize_response_with_error(response, prompt):
    """
    如果response在前几轮有错误，重组prompt和response
    把错误前的部分拼接到prompt，错误及之后作为新response
    返回: (new_prompt, new_response)
    """
    observations = extract_observations(response)
    
    # 从prompt中提取system\n和\nassistant之间的内容
    prompt_match = re.search(r'system\n(.*?)\nassistant', prompt, re.DOTALL)
    if prompt_match:
        prompt = prompt_match.group(1).strip()
    
    if len(observations) == 0:
        return (prompt, response)
    
    # 查找最后一个有错误的observation
    last_error_idx = -1
    for i, obs in enumerate(observations):
        if has_error_in_observation(obs):
            last_error_idx = i
    
    # 如果没有错误，直接返回原样
    if last_error_idx == -1:
        return (prompt, response)
    
    # 如果最后一轮还是错误，直接返回None（说明没有成功的尝试）
    if last_error_idx == len(observations) - 1:
        return None
    
    # 找到最后一个错误observation在response中的位置
    # 需要找到 "Code execution result: {error_obs}" 的位置
    error_obs = observations[last_error_idx]
    pattern = r'Code execution result:\s*' + re.escape(error_obs)
    match = re.search(pattern, response)
    
    if not match:
        return (prompt, response)
    
    # 切分点：从错误observation之后开始（不包含错误observation本身）
    split_pos = match.end()
    
    # 前半部分：prompt + 截止到错误observation结束的response（包含错误）
    before_and_including_error = response[:split_pos].strip()
    new_prompt = prompt + "\n\n" + before_and_including_error
    
    # 后半部分：从错误observation之后的内容（不包含错误observation）
    after_error = response[split_pos:].strip()
    
    return (new_prompt, after_error)

print("Test reorganize function...")
test_resp = """```python
x = 1/0
```

Code execution result: ZeroDivisionError: division by zero

Let me try again:

```python
x = 5
print(x)
```

Code execution result: 5"""

test_prompt = "Solve the problem:"
new_prompt, new_response = reorganize_response_with_error(test_resp, test_prompt)

print(f"\n原始 Prompt: {test_prompt}")
print(f"\n重组后 Prompt: {new_prompt[:200]}...")
print(f"\n重组后 Response: {new_response[:200]}...")


Test reorganize function...

原始 Prompt: Solve the problem:

重组后 Prompt: Solve the problem:

```python
x = 1/0
```

Code execution result: ZeroDivisionError: division by zero...

重组后 Response: Let me try again:

```python
x = 5
print(x)
```

Code execution result: 5...


In [9]:
# 处理所有responses，重组有错误的样本
def count_turns(response):
    """统计response中call tool的轮数"""
    return len(extract_observations(response))

# 处理所有responses，对有错误的进行重组
all_processed_samples = []

reorganized_count = 0
for prompt_idx, responses in prompt2response_no_error.items():
    for resp_info in responses:
        response = resp_info['response']
        prompt = resp_info['prompt']
        score = resp_info['score']
        
        # 检查是否有错误并重组
        result = reorganize_response_with_error(response, prompt)
        prompt_match = re.search(r'system\n(.*?)\nassistant', prompt, re.DOTALL)
        if prompt_match:
            new_prompt = prompt_match.group(1).strip()
        
        # 如果返回None（最后一轮还是错误，无法重组），跳过这个样本
        if result is None:
            continue
        
        new_response = response
        
        # 统计是否被重组
        if new_prompt != prompt:
            reorganized_count += 1
        
        all_processed_samples.append({
            'prompt_idx': prompt_idx,
            'prompt': new_prompt,
            'response': new_response,
            'score': score,
            'turns': count_turns(new_response),
            'reorganized': new_prompt != prompt
        })

print(f"Total processed samples: {len(all_processed_samples)}")
print(f"Reorganized samples (有错误被重组的): {reorganized_count}")

# 按prompt_idx分组
from collections import defaultdict
prompt2samples = defaultdict(list)
for sample in all_processed_samples:
    prompt2samples[sample['prompt_idx']].append(sample)

print(f"Total prompts: {len(prompt2samples)}")


Total processed samples: 179573
Reorganized samples (有错误被重组的): 179573
Total prompts: 7904


In [10]:
# 为每个prompt选择两条样本：尽量一条单轮，一条多轮
final_filtered_samples = []

single_turn_count = 0
multi_turn_count = 0
only_one_sample_count = 0
same_turn_count = 0

for prompt_idx, samples in prompt2samples.items():
    if len(samples) == 0:
        continue
    elif len(samples) == 1:
        # 只有一条，直接保留
        final_filtered_samples.append(samples[0])
        only_one_sample_count += 1
    else:
        # 多条样本，选择一条单轮和一条多轮
        # 分类：单轮(turns==1)和多轮(turns>1)
        single_turn_samples = [s for s in samples if s['turns'] == 1]
        multi_turn_samples = [s for s in samples if s['turns'] > 1]
        
        selected = []
        
        if len(single_turn_samples) > 0 and len(multi_turn_samples) > 0:
            # 理想情况：有单轮也有多轮
            selected.append(single_turn_samples[0])
            # 按轮数排序，选择轮数最少的多轮样本（优先2轮，然后3轮等）
            multi_turn_samples_sorted = sorted(multi_turn_samples, key=lambda x: x['turns'])
            selected.append(multi_turn_samples_sorted[0])
            single_turn_count += 1
            multi_turn_count += 1
        elif len(single_turn_samples) >= 2:
            # 只有单轮，选两条
            selected.append(single_turn_samples[0])
            selected.append(single_turn_samples[1])
            same_turn_count += 1
        elif len(multi_turn_samples) >= 2:
            # 只有多轮，选两条（按轮数排序，优先选轮数少的）
            multi_turn_samples_sorted = sorted(multi_turn_samples, key=lambda x: x['turns'])
            selected.append(multi_turn_samples_sorted[0])
            # 尝试找一个turn数不同的
            found_diff = False
            for s in multi_turn_samples_sorted[1:]:
                if s['turns'] != multi_turn_samples_sorted[0]['turns']:
                    selected.append(s)
                    found_diff = True
                    break
            if not found_diff:
                selected.append(multi_turn_samples_sorted[1])
            same_turn_count += 1
        else:
            # 只有一种类型且只有一条，保留这一条
            selected.append(samples[0])
            only_one_sample_count += 1
        
        final_filtered_samples.extend(selected)

print(f"\n筛选结果:")
print(f"  总样本数: {len(final_filtered_samples)}")
print(f"  成功配对(一单一多): {single_turn_count} prompts")
print(f"  相同轮数的配对: {same_turn_count} prompts")
print(f"  只有一条样本: {only_one_sample_count} prompts")

# 统计turn分布
from collections import Counter
turn_distribution = Counter([s['turns'] for s in final_filtered_samples])
print(f"\nTurn分布:")
for turns, count in sorted(turn_distribution.items()):
    print(f"  {turns} turns: {count} samples")



筛选结果:
  总样本数: 15425
  成功配对(一单一多): 4976 prompts
  相同轮数的配对: 2545 prompts
  只有一条样本: 383 prompts

Turn分布:
  1 turns: 9798 samples
  2 turns: 5024 samples
  3 turns: 487 samples
  4 turns: 107 samples
  5 turns: 9 samples


In [11]:
# 查看一些被重组的样本示例
print("查看被重组的样本示例（前3个）:\n")
print("=" * 100)

reorganized_shown = 0
for prompt_idx, samples in prompt2samples.items():
    # 检查是否有被重组的样本
    reorganized_samples = [s for s in samples if s.get('reorganized', False)]
    
    if len(reorganized_samples) > 0:
        reorganized_shown += 1
        if reorganized_shown <= 3:
            print(f"\nPrompt Index: {prompt_idx}")
            print(f"该prompt有 {len(reorganized_samples)} 个样本被重组\n")
            
            for i, sample in enumerate(reorganized_samples[:2]):  # 最多显示2个
                print(f"--- 重组样本 {i+1} (Turns: {sample['turns']}) ---")
                print(f"新Prompt:\n{sample['prompt']}...\n")
                print(f"新Response:\n{sample['response']}...\n")
                print("-" * 80)
        
        if reorganized_shown >= 3:
            break

print(f"\n总共有 {reorganized_count} 个样本被重组（prompt变长了）")


查看被重组的样本示例（前3个）:


Prompt Index: 35379
该prompt有 13 个样本被重组

--- 重组样本 1 (Turns: 1) ---
新Prompt:
You are a helpful assistant.
user
Solve the following problem step by step. You now have the ability to selectively write executable Python code to enhance your reasoning process. The Python code will be executed by an external sandbox, and the output (after "Code execution result: ") is returned to aid your reasoning and help you arrive at the final answer. The Python code should be complete scripts, including necessary imports.

Code Format:
Each code snippet is wrapped between ```. You need to use `print()` to output intermediate results.

Answer Format:
You must use \boxed to return your answer. The last part of your response should be:
\boxed{'The final answer goes here.'}

User Question:
Automobile license plates for a state consist of four letters followed by a dash and two single digits. How many different license plate combinations are possible if exactly one letter is repeated exactl

In [12]:
# 验证所有最终样本是否有效
print("验证最终样本...")
invalid_count = 0
for sample in final_filtered_samples:
    if not is_valid_response(sample['response']):
        invalid_count += 1
        if invalid_count <= 3:
            print(f"\n发现无效样本 (Prompt {sample['prompt_idx']}):")
            print(f"Response: {sample['response'][:500]}...")
            print("-" * 80)

print(f"\n无效样本数: {invalid_count} / {len(final_filtered_samples)}")
print(f"有效样本数: {len(final_filtered_samples) - invalid_count}")


验证最终样本...

无效样本数: 0 / 15425
有效样本数: 15425


In [14]:
final_filtered_samples[3]

{'prompt_idx': 14344,
 'prompt': 'You are a helpful assistant.\nuser\nSolve the following problem step by step. You now have the ability to selectively write executable Python code to enhance your reasoning process. The Python code will be executed by an external sandbox, and the output (after "Code execution result: ") is returned to aid your reasoning and help you arrive at the final answer. The Python code should be complete scripts, including necessary imports.\n\nCode Format:\nEach code snippet is wrapped between ```. You need to use `print()` to output intermediate results.\n\nAnswer Format:\nYou must use \\boxed to return your answer. The last part of your response should be:\n\\boxed{\'The final answer goes here.\'}\n\nUser Question:\nAmong the 2019 natural numbers from 1 to 2019, how many of them, when added to the four-digit number 8866, result in at least one carry?',
 'response': 'To solve this problem, we need to check how many of the numbers from 1 to 2019, when added to 

In [15]:
sft_data = []
for sample in final_filtered_samples:
    # 使用 re.DOTALL 让 . 匹配换行符，正确提取多行内容
    # 提取\nuser\n之前的内容作为system_prompt（如果存在）
    system_match = re.search(r'^(.*?)\nuser\n', sample['prompt'], re.DOTALL)
    # 提取user\n之后的内容（去掉末尾的\nassistant如果存在）
    user_match = re.search(r'user\n(.*?)(?:\nassistant)?$', sample['prompt'], re.DOTALL)
    
    if user_match:
        system_prompt = system_match.group(1).strip() if system_match else ""
        user_prompt = user_match.group(1).strip()
        
        messages = []
        if system_prompt:
            messages.append({"role": "system", "content": system_prompt})
        messages.append({"role": "user", "content": user_prompt})
        messages.append({"role": "assistant", "content": sample['response']})
        
        sft_data.append({"messages": messages})

In [16]:
sft_data[1]

{'messages': [{'role': 'system', 'content': 'You are a helpful assistant.'},
  {'role': 'user',
   'content': 'Solve the following problem step by step. You now have the ability to selectively write executable Python code to enhance your reasoning process. The Python code will be executed by an external sandbox, and the output (after "Code execution result: ") is returned to aid your reasoning and help you arrive at the final answer. The Python code should be complete scripts, including necessary imports.\n\nCode Format:\nEach code snippet is wrapped between ```. You need to use `print()` to output intermediate results.\n\nAnswer Format:\nYou must use \\boxed to return your answer. The last part of your response should be:\n\\boxed{\'The final answer goes here.\'}\n\nUser Question:\nAutomobile license plates for a state consist of four letters followed by a dash and two single digits. How many different license plate combinations are possible if exactly one letter is repeated exactly o

In [38]:
import json

with open('filtered_sft_data.jsonl', 'w') as f:
    for sample in sft_data:
        f.write(json.dumps(sample) + '\n')



In [17]:
from datasets import load_dataset
filtered_sft_data = load_dataset("json", data_files="filtered_sft_data.jsonl", split="train")
len(filtered_sft_data)

15383

In [18]:
import re
from transformers import AutoTokenizer

# 加载tokenizer用于计算token数
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B", trust_remote_code=False)

truncated_count = 0
processed_data = []

for sample in filtered_sft_data:
    # 获取assistant的回复内容（messages列表的最后一个元素）
    messages = sample['messages']
    assistant_content = messages[-1]['content']
    
    # 找到最后一个 \boxed{} 的位置
    # 使用正则表达式匹配 \boxed{...}，考虑嵌套的大括号
    boxed_pattern = r'\\boxed\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}'
    matches = list(re.finditer(boxed_pattern, assistant_content))
    
    if matches:
        # 获取最后一个 \boxed{} 的结束位置
        last_boxed_end = matches[-1].end()
        
        # 提取 \boxed{} 之后的内容
        after_boxed = assistant_content[last_boxed_end:]
        
        # 计算 \boxed{} 之后的token数
        after_boxed_tokens = tokenizer.encode(after_boxed, add_special_tokens=False)
        
        # 如果超过300个token，截断
        if len(after_boxed_tokens) > 300:
            print(f"截断的样本: {sample['messages'][-1]['content']}")
            truncated_count += 1
            continue
    
    processed_data.append({'messages': messages})

print(f"总样本数: {len(filtered_sft_data)}")
print(f"被截断的样本数: {truncated_count}")
print(f"截断比例: {truncated_count/len(filtered_sft_data)*100:.2f}%")


总样本数: 15383
被截断的样本数: 0
截断比例: 0.00%


In [19]:
len(processed_data)

15383

In [20]:
# 保存处理后的数据
import json

with open('filtered_sft_data.jsonl', 'w') as f:
    for sample in processed_data:
        f.write(json.dumps(sample) + '\n')

print(f"已保存处理后的 {len(processed_data)} 条样本到 filtered_sft_data.jsonl")


已保存处理后的 15383 条样本到 filtered_sft_data.jsonl
